# Aggregate Functions Questions



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

## Que 01: Duplicate Emails Detection

Find all emails that appear more than once in the database.

**Table:** `person`

| Column | Description |
|--------|-------------|
| id | Person ID |
| email | Email address |

**Output**

Return all distinct emails that appear **2 or more times**.

**Requirements**
- Return distinct duplicate emails.
- Sort the result alphabetically.
- Result column name: `email`.

In [0]:
person_data = [(1,"alice@example.com"),(2,"bob@example.com"),(3,"alice@example.com"),(4,"charlie@example.com"),(5,"bob@example.com")]
person_df = spark.createDataFrame(person_data,["id","email"])


count_df = person_df.groupBy("email").agg(count("email").alias("count"))

output_df = count_df.select("email").where("count > 1").orderBy("email")

display(output_df)

## Que 02: Top 5 States by 5-Star Businesses

**Difficulty:** EASY

### Problem

You have a businesses table with star ratings. Count how many businesses have a 5-star rating in each state, then return the top 5 states by count.

Return `state` and `five_star_count` for the top 5 states. Sorted by `five_star_count` descending.

**Schema columns:** `businesses.business_id`, `businesses.name`, `businesses.state`, `businesses.stars`, `businesses.review_count`

**Output columns:** `state`, `five_star_count`

**Sort by** `five_star_count` descending, then by `state` ascending for ties.

### Schema

#### businesses

| Column | Type |
|--------|------|
| business_id | |
| name | |
| state | |
| stars | |
| review_count | |

### Example

**Input**

| business_id | name | state | stars | review_count |
|------------:|------|-------|------:|-------------:|
| 1 | Restaurant A | CA | 5.0 | 150 |
| 2 | Cafe B | CA | 4.5 | 100 |
| 3 | Shop C | CA | 5.0 | 200 |
| 4 | Store D | NY | 5.0 | 120 |
| 5 | Market E | NY | 4.0 | 180 |

**Output**

| state | five_star_count |
|-------|----------------:|
| TX | 3 |
| CA | 2 |
| WA | 2 |
| FL | 1 |
| NY | 1 |

**Explanation**

The output shows 5 row(s) derived by applying the required transformations to the input data.

### Constraints

- Only count businesses with `stars = 5.0` exactly.
- Return top 5 states (or fewer if fewer than 5 states have 5-star businesses).
- Sort by `five_star_count` descending, then by `state` ascending for ties.
- State is never null.

In [0]:
businesses_data = [(1,"Restaurant A","CA",5.0,150), (2,"Cafe B","CA",4.5,100), (3,"Shop C","CA",5.0,200), (4,"Store D","NY",5.0,120), (5,"Market E","NY",4.0,180), (6,"BBQ House","TX",5.0,250), (7,"Pizza Hub","TX",5.0,175), (8,"Coffee Spot","TX",5.0,90), (9,"Bakery","WA",5.0,110), (10,"Diner","WA",5.0,140), (11,"Grill","FL",5.0,130)]

businesses_df = spark.createDataFrame(businesses_data, ["business_id","name","state","stars","review_count"])

output_df = (
businesses_df
.where("stars = 5")
.groupBy("state")
.agg(count("business_id").alias("five_star_count"))
.orderBy(col("five_star_count").desc(), col("state").asc())
.limit(5)
)

display(output_df)


state,five_star_count
TX,3
CA,2
WA,2
FL,1
NY,1


## Que 03: Duplicate Email Records

**Difficulty:** EASY

### Problem

A data quality team needs to identify duplicate email addresses in the user database to prevent account conflicts and improve data integrity.

**Schema columns:** `users.user_id`, `users.name`, `users.email`

**Output columns:** `email`, `occurrence_count`

**Sort by** the first column in ascending order.

### Schema

#### users

| user_id | name | email |
|---------|------|-------|

### Example

**Input**

| user_id | name | email |
|---------|--------|------------------|
| 1 | User_1 | john@example.com |
| 2 | User_2 | jane@example.com |
| 3 | User_3 | bob@example.com |
| 4 | User_4 | john@example.com |
| 5 | User_5 | jane@example.com |
| 6 | User_6 | bob@example.com |
| 7 | User_7 | alice@example.com |
| 8 | User_8 | mike@example.com |

**Output**

| email | occurrence_count |
|------------------|-----------------:|
| bob@example.com | 2 |
| jane@example.com | 2 |
| john@example.com | 2 |

**Explanation**

From the 8 rows in the input table(s), 3 rows match the query criteria and are returned in the output.

### Constraints

- Only include emails that appear more than once (`count > 1`).
- Order by `occurrence_count` descending, then by `email` ascending.
- Use aggregation with HAVING clause.
- Single email column and count column in output.

In [0]:
users_data = [(1,"User_1","john@example.com"),(2,"User_2","jane@example.com"),(3,"User_3","bob@example.com"),(4,"User_4","john@example.com"),(5,"User_5","jane@example.com"),(6,"User_6","bob@example.com"),(7,"User_7","alice@example.com"),(8,"User_8","mike@example.com")]
users_df = spark.createDataFrame(users_data,["user_id","name","email"])


output_df = (
users_df.groupBy("email").agg(count("email").alias("occurence_count"))
.where("occurence_count > 1")
.orderBy(col("occurence_count").desc(), col("email").asc())
)

display(output_df)



email,occurence_count
bob@example.com,2
jane@example.com,2
john@example.com,2


## Que 04: Venture-Funded Startup Analysis

**Difficulty:** MEDIUM

### Problem

Which VC firm is paying the biggest premium over its own funding ceiling?

You are a data analyst at Sequoia studying how competing venture firms behave against their internal funding limits. Each firm sets a per-startup funding ceiling, and the portfolio team wants to flag the firm whose average check size most clearly exceeds its own limit, ignoring firms with too few deals to judge.

Write a query that joins `fs_venture_capitalist` to `fs_funded_startups` on `vc_id` and, for each firm, computes the average funding across its startups, kept to **2 decimal places** as `avg_funding`. Keep only firms where this average is **strictly greater** than their `funding_limit` and that have funded **at least 2 startups**. Sort the qualifying firms by `avg_funding` in descending order and return only the single top firm. Also return `funding_limit` to **2 decimal places**.

**Schema columns:** `fs_funded_startups.startup_id`, `fs_funded_startups.startup_name`, `fs_funded_startups.vc_id`, `fs_funded_startups.funding`, `fs_venture_capitalist.vc_id`, `fs_venture_capitalist.vc_name`, `fs_venture_capitalist.funding_limit`

**Output columns:** `avg_funding`, `funding_limit`, `vc_id`, `vc_name`


### Example

**Input**

**fs_funded_startups**

| startup_id | startup_name | vc_id | funding |
|------------|--------------|-------|---------|
| S1 | Startup 1 | VC1 | 2 |
| S2 | Startup 2 | VC1 | 1 |
| S3 | Startup 3 | VC2 | 2.5 |
| S4 | Startup 4 | VC2 | 2 |
| S5 | Startup 5 | VC3 | 1.8 |
| S6 | Startup 6 | VC3 | 1.7 |

**fs_venture_capitalist**

| vc_id | vc_name | funding_limit |
|-------|----------|---------------|
| VC1 | VC Firm 1 | 1.5 |
| VC2 | VC Firm 2 | 2 |
| VC3 | VC Firm 3 | 1.75 |
| VC4 | VC Firm 4 | 2.5 |

**Output**

| avg_funding | funding_limit | vc_id | vc_name |
|-------------|---------------|-------|----------|
| 2.25 | 2 | VC2 | VC Firm 2 |

**Explanation**

VC2 funded two startups averaging `(2.5 + 2) / 2 = 2.25`, strictly above its `2.00` limit, so it qualifies. VC1's average of `1.5` only equals its `1.5` limit and is excluded, as is any firm whose average does not strictly exceed its ceiling or that funded fewer than 2 startups; among qualifiers, only the firm with the highest `avg_funding` is returned. The `fs_funded_startups` table contains more rows than sampled above; the output is computed over the full dataset.

### Constraints

- Join `fs_venture_capitalist` to `fs_funded_startups` on `vc_id` (inner join); firms with no funded startups never qualify.
- A firm qualifies only if its average funding is strictly greater than its `funding_limit` (equal does not qualify).
- A firm qualifies only if it has funded at least 2 startups.
- `avg_funding` and `funding_limit` must be reported to 2 decimal places.
- Output columns must be exactly `avg_funding`, `funding_limit`, `vc_id`, `vc_name`.
- Sort qualifying firms by `avg_funding` in descending order and return only the top 1 row.

In [0]:
fs_funded_startups_data = [("S1","Startup 1","VC1",2.0),("S2","Startup 2","VC1",1.0),("S3","Startup 3","VC2",2.5),("S4","Startup 4","VC2",2.0),("S5","Startup 5","VC3",1.8),("S6","Startup 6","VC3",1.7)]
fs_funded_startups_df = spark.createDataFrame(fs_funded_startups_data,["startup_id","startup_name","vc_id","funding"])

fs_venture_capitalist_data = [("VC1","VC Firm 1",1.5),("VC2","VC Firm 2",2.0),("VC3","VC Firm 3",1.75),("VC4","VC Firm 4",2.5)]
fs_venture_capitalist_df = spark.createDataFrame(fs_venture_capitalist_data,["vc_id","vc_name","funding_limit"])


grouped_df = (fs_funded_startups_df
.groupBy("vc_id")
.agg(count("startup_id").alias("count"), round(avg("funding"), 2).alias("avg_funding"))
.where("count >= 2")
)

joined_df = grouped_df.join(fs_venture_capitalist_df, on="vc_id", how="inner").where("avg_funding > funding_limit")

output_df = joined_df.select("avg_funding", "funding_limit", "vc_id", "vc_name").orderBy(col("avg_funding").desc()).limit(1)

display(output_df)


avg_funding,funding_limit,vc_id,vc_name
2.25,2.0,VC2,VC Firm 2


## Que 05: Credit Card Launch Analysis

**Difficulty:** MEDIUM

### Problem

Your team at a leading financial institution is preparing to launch a new credit card. To estimate how many cards will be issued in the first month, you need to analyze how previous credit card launches performed during their debut months. Write a query that retrieves the name of each credit card and the number of cards issued in its launch month.

**Schema columns:** `ccd_credit_cards.issue_month`, `ccd_credit_cards.issue_year`, `ccd_credit_cards.card_name`, `ccd_credit_cards.issued_amount`

**Output columns:** `card_name`, `issued_amount`

**Sort the results by** `issued_amount DESC`.

### Schema

#### ccd_credit_cards

| issue_month | issue_year | card_name | issued_amount |
|-------------|------------|-----------|---------------|

### Example

**Input**

| issue_month | issue_year | card_name | issued_amount |
|-------------|-------------|----------------|--------------:|
| 1 | 2021 | Sapphire Plus | 170000 |
| 2 | 2021 | Sapphire Plus | 175000 |
| 3 | 2021 | Sapphire Plus | 180000 |
| 3 | 2021 | Freedom Flex | 65000 |

**Output**

| card_name | issued_amount |
|-----------|--------------:|
| Sapphire Plus | 170000 |
| Freedom Flex | 65000 |

### Constraints

- Handle NULL values appropriately.
- Return results matching the expected output schema and order.

In [0]:
ccd_credit_cards_data = [(1,2021,"Sapphire Plus",170000),(2,2021,"Sapphire Plus",175000),(3,2021,"Sapphire Plus",180000),(3,2021,"Freedom Flex",65000)]
ccd_credit_cards_df = spark.createDataFrame(ccd_credit_cards_data,["issue_month","issue_year","card_name","issued_amount"])
display(ccd_credit_cards_df)

grouped_df = (ccd_credit_cards_df
.groupBy("card_name", "issue_year")
.agg(min(col("issue_month")).alias("min_issue_month"))
)

joining_condition = (
    (col("g.card_name") == col("c.card_name")) & 
    (col("g.issue_year") == col("c.issue_year")) & 
    (col("g.min_issue_month") == col("c.issue_month")) 
)

joined_df = grouped_df.alias("g").join(ccd_credit_cards_df.alias("c"), on=joining_condition, how="inner")

output_df = joined_df.select("c.card_name", "c.issued_amount")

display(output_df)

issue_month,issue_year,card_name,issued_amount
1,2021,Sapphire Plus,170000
2,2021,Sapphire Plus,175000
3,2021,Sapphire Plus,180000
3,2021,Freedom Flex,65000


card_name,issued_amount
Sapphire Plus,170000
Freedom Flex,65000


## Que 06: Peak Energy Usage Period

**MEDIUM**

### Problem

A data-center operator tracks daily energy consumption for Asia, Europe, and North America. Combine consumption recorded on the same date across all regions and return the date with the highest total. If multiple dates tie, return the earliest date.

**Schema columns:** `pec_asia_energy.date`, `pec_asia_energy.consumption`, `pec_eu_energy.date`, `pec_eu_energy.consumption`, `pec_na_energy.date`, `pec_na_energy.consumption`

**Output columns:** `date`, `total_consumption`
### Examples

#### Example 1

**Input:**

**pec_asia_energy:**

| date | consumption |
|------|------------:|
| 2020-01-01 | 400 |
| 2020-01-02 | 400 |
| 2020-01-04 | 675 |
| 2020-01-05 | 1200 |

**pec_eu_energy:**

| date | consumption |
|------|------------:|
| 2020-01-01 | 400 |
| 2020-01-02 | 350 |
| 2020-01-03 | 500 |
| 2020-01-04 | 500 |

**pec_na_energy:**

| date | consumption |
|------|------------:|
| 2020-01-01 | 250 |
| 2020-01-02 | 375 |
| 2020-01-03 | 600 |
| 2020-01-06 | 500 |

**Output:**

| date | total_consumption |
|------|------------------:|
| 2020-01-05 | 1200 |

**Explanation:** January 5 has the greatest total among the rows shown, with 1200 from Asia.

### Constraints

- A date may appear in any subset of the three regional tables.
- Resolve equal totals by choosing the earliest date.
- Return results matching the expected output schema and order.

In [0]:
pec_asia_energy_data = [("2020-01-01",400),("2020-01-02",400),("2020-01-04",675),("2020-01-05",1200)]
pec_asia_energy_df = spark.createDataFrame(pec_asia_energy_data,["date","consumption"])

pec_eu_energy_data = [("2020-01-01",400),("2020-01-02",350),("2020-01-03",500),("2020-01-04",500)]
pec_eu_energy_df = spark.createDataFrame(pec_eu_energy_data,["date","consumption"])

pec_na_energy_data = [("2020-01-01",250),("2020-01-02",375),("2020-01-03",600),("2020-01-06",500)]
pec_na_energy_df = spark.createDataFrame(pec_na_energy_data,["date","consumption"])


merged_df = pec_asia_energy_df.unionAll(pec_eu_energy_df).unionAll(pec_na_energy_df)

grouped_df = (merged_df
.groupBy("date").agg(sum("consumption").alias("total_consumption"))
.orderBy(col("total_consumption").desc(), col("date").asc())
)

output_df = grouped_df.limit(1)

display(output_df)



date,total_consumption
2020-01-05,1200


## Que 07: PySpark Filter and Map Transformations for User Engagement

**MEDIUM**

### Problem

You have a table of user events with activity type, date, duration, and active status. Calculate engagement metrics for only active users.

Filter to include only active users (`is_active = 1`), aggregate events per user, and calculate an engagement score based on event count and duration. Return results sorted by engagement score in descending order.

**Schema columns:** `user_events.user_id`, `user_events.event_type`, `user_events.event_date`, `user_events.duration_seconds`, `user_events.is_active`

**Output columns:** `user_id`, `total_events`, `total_duration`, `avg_duration`, `engagement_score`

Order the result by `engagement_score DESC`.

### Examples

#### Example 1

**Input:**

**user_events:**

| user_id | event_type | event_date | duration_seconds | is_active |
|---------|------------|------------|-----------------:|----------:|
| U001 | login | 2024-01-15 | 45 | 1 |
| U001 | view | 2024-01-16 | 120 | 1 |
| U001 | purchase | 2024-01-17 | 60 | 1 |
| U002 | login | 2024-01-15 | 30 | 1 |
| U002 | view | 2024-01-16 | 90 | 1 |

**Output:**

| user_id | total_events | total_duration | avg_duration | engagement_score |
|---------|-------------:|---------------:|-------------:|-----------------:|
| U004 | 4 | 265 | 66.25 | 4.25 |
| U007 | 3 | 260 | 86.67 | 3.8 |
| U001 | 3 | 225 | 75.0 | 3.45 |
| U010 | 3 | 205 | 68.33 | 3.25 |
| U005 | 3 | 115 | 38.33 | 2.35 |

**Explanation:** The output is derived by applying the required transformations to the input data according to the problem statement.

### Constraints

- Only include users where `is_active = 1`
- Engagement score = `(total_events * 0.4 + total_duration_in_minutes * 0.6)`
- Round all numeric outputs to 2 decimal places
- Order results by `engagement_score` in descending order
- Duration is measured in seconds and should be converted to minutes for the engagement score calculation
- Inactive users should be completely excluded from the result set

In [0]:
user_events_data = [("U001","login","2024-01-15",45,1),("U001","view","2024-01-16",120,1),("U001","purchase","2024-01-17",60,1),("U002","login","2024-01-15",30,1),("U002","view","2024-01-16",90,1),("U004","login","2024-01-15",50,1),("U004","view","2024-01-16",65,1),("U004","purchase","2024-01-17",70,1),("U004","logout","2024-01-18",80,1),("U005","login","2024-01-15",30,1),("U005","view","2024-01-16",35,1),("U005","purchase","2024-01-17",50,1),("U007","login","2024-01-15",80,1),("U007","view","2024-01-16",90,1),("U007","purchase","2024-01-17",90,1),("U010","login","2024-01-15",65,1),("U010","view","2024-01-16",70,1),("U010","purchase","2024-01-17",70,1),("U003","login","2024-01-15",40,0),("U006","view","2024-01-16",100,0)]
user_events_df = spark.createDataFrame(user_events_data,["user_id","event_type","event_date","duration_seconds","is_active"])

display(user_events_df)


user_events_df = user_events_df.filter(col("is_active") == 1)

grouped_df = (user_events_df.
groupBy("user_id").
agg(
    round(count("user_id"), 2).alias("total_events"),
    round(sum("duration_seconds"), 2).alias("total_duration"),
    round(avg("duration_seconds"), 2).alias("avg_duration")
))

output_df = (grouped_df.
withColumn("Engagement_score", round(col("total_events") * 0.4 + col("total_duration") / 60 * 0.6, 2)).
orderBy(col("Engagement_score").desc()
))

display(output_df)


user_id,event_type,event_date,duration_seconds,is_active
U001,login,2024-01-15,45,1
U001,view,2024-01-16,120,1
U001,purchase,2024-01-17,60,1
U002,login,2024-01-15,30,1
U002,view,2024-01-16,90,1
U004,login,2024-01-15,50,1
U004,view,2024-01-16,65,1
U004,purchase,2024-01-17,70,1
U004,logout,2024-01-18,80,1
U005,login,2024-01-15,30,1


user_id,total_events,total_duration,avg_duration,Engagement_score
U004,4,265,66.25,4.25
U007,3,260,86.67,3.8
U001,3,225,75.0,3.45
U010,3,205,68.33,3.25
U005,3,115,38.33,2.35
U002,2,120,60.0,2.0


## Que 08: Merchant Settlement Analysis

**MEDIUM**

### Problem

A payment settlement system needs to calculate daily settlement amounts per merchant, accounting for sales, refunds, and fund holds/releases.

For each merchant and date, calculate:

- gross_sales (sum of 'sale' txn_type amounts)
- refunds (sum of 'refund' txn_type amounts)
- net_holds (sum of 'hold' amounts minus sum of 'release' amounts)
- settlement_amount = gross_sales - refunds - net_holds

**Schema columns:** `transactions.txn_id`, `transactions.merchant_id`, `transactions.txn_type`, `transactions.amount`, `transactions.txn_date`

**Output columns:** `merchant_id`, `txn_date`, `gross_sales`, `refunds`, `net_holds`, `settlement_amount`

Order the result by `merchant_id`, `txn_date`.

### Schema

#### transactions

| txn_id | merchant_id | txn_type | amount | txn_date |
|--------|-------------|----------|--------|----------|

### Examples

#### Example 1

**Input:**

**transactions:**

| txn_id | merchant_id | txn_type | amount | txn_date |
|--------|-------------|----------|-------:|----------|
| 1001 | 201 | sale | 100 | 2024-01-15 |
| 1002 | 202 | sale | 250 | 2024-01-15 |
| 1003 | 201 | sale | 150 | 2024-01-15 |
| 1004 | 203 | sale | 75 | 2024-01-15 |
| 1005 | 201 | sale | 200 | 2024-01-15 |

**Output:**

| merchant_id | txn_date | gross_sales | refunds | net_holds | settlement_amount |
|------------:|----------|------------:|--------:|----------:|------------------:|
| 201 | 2024-01-15 | 1150 | 0 | 0 | 1150 |
| 201 | 2024-01-16 | 800 | 175 | 0 | 625 |
| 201 | 2024-01-17 | 0 | 0 | -200 | 200 |
| 202 | 2024-01-15 | 1000 | 0 | 0 | 1000 |
| 202 | 2024-01-16 | 500 | 150 | 0 | 350 |

**Explanation:** The output is derived by applying the required transformations to the input data according to the problem statement.

### Constraints

- Handles merchants with no sales on a day
- Handles days with only refunds

In [0]:
transactions_data = [(1001,201,"sale",100,"2024-01-15"),(1002,202,"sale",250,"2024-01-15"),(1003,201,"sale",150,"2024-01-15"),(1004,203,"sale",75,"2024-01-15"),(1005,201,"sale",200,"2024-01-15"),(1006,201,"sale",700,"2024-01-15"),(1007,202,"sale",750,"2024-01-15"),(1008,201,"sale",800,"2024-01-16"),(1009,201,"refund",175,"2024-01-16"),(1010,202,"sale",500,"2024-01-16"),(1011,202,"refund",150,"2024-01-16"),(1012,201,"release",200,"2024-01-17")]
transactions_df = spark.createDataFrame(transactions_data,["txn_id","merchant_id","txn_type","amount","txn_date"])
display(transactions_df)


grouped_df = (transactions_df.
groupBy("merchant_id", "txn_date").
agg(
    sum(when(col("txn_type") == "sale", col("amount")).otherwise(0)).alias("gross_sales"),
    sum(when(col("txn_type") == "refund", col("amount")).otherwise(0)).alias("refunds"),
    sum(when(col("txn_type") == "hold", col("amount")).otherwise(0)).alias("holds"),
    sum(when(col("txn_type") == "release", col("amount")).otherwise(0)).alias("releases"),
))

# settlement_amount = gross_sales - refunds - net_holds
output_df = (grouped_df.
withColumn("net_holds", col("holds") - col("releases")).
withColumn("settlement_amount", col("gross_sales") - col("refunds") - col("net_holds"))
)

output_df = output_df.select("merchant_id",	"txn_date",	"gross_sales",	"refunds", "net_holds", "settlement_amount").orderBy("merchant_id", "txn_date")

display(output_df)


txn_id,merchant_id,txn_type,amount,txn_date
1001,201,sale,100,2024-01-15
1002,202,sale,250,2024-01-15
1003,201,sale,150,2024-01-15
1004,203,sale,75,2024-01-15
1005,201,sale,200,2024-01-15
1006,201,sale,700,2024-01-15
1007,202,sale,750,2024-01-15
1008,201,sale,800,2024-01-16
1009,201,refund,175,2024-01-16
1010,202,sale,500,2024-01-16


merchant_id,txn_date,gross_sales,refunds,net_holds,settlement_amount
201,2024-01-15,1150,0,0,1150
201,2024-01-16,800,175,0,625
201,2024-01-17,0,0,-200,200
202,2024-01-15,1000,0,0,1000
202,2024-01-16,500,150,0,350
203,2024-01-15,75,0,0,75


## Que 09: Identify VIP Customers

### Problem

Identify VIP customers from a ride-sharing platform. A customer is considered VIP if they have completed 50 or more rides AND spent $500 or more in total.

**Schema columns:**

`rides.ride_id`, `rides.user_id`, `rides.ride_cost`, `rides.ride_date`, `rides.distance`, `users.user_id`, `users.name`, `users.signup_date`

**Output columns:** `user_id`, `name`, `total_rides`, `total_spent`

Sort the results by `total_spent DESC`.

### Examples

#### Example 1

**Input:**

**rides:**

| ride_id | user_id | ride_cost | ride_date | distance |
|---------|---------|----------:|-----------|---------:|
| 1 | 1 | 15.50 | 2023-01-05 | 8.5 |
| 2 | 1 | 12.30 | 2023-01-08 | 6.2 |
| 3 | 1 | 18.75 | 2023-01-12 | 9.3 |
| 4 | 1 | 14.20 | 2023-01-15 | 7.1 |

**users:**

| user_id | name | signup_date |
|---------|--------------|------------|
| 1 | Alice Johnson | 2022-01-15 |
| 2 | Bob Smith | 2021-06-20 |
| 3 | Carol White | 2022-03-10 |
| 4 | David Brown | 2021-12-05 |

**Output:**

| user_id | name | total_rides | total_spent |
|---------|--------------|------------:|------------:|
| 5 | Eve Davis | 60 | 1263.9 |
| 3 | Carol White | 55 | 1213.25 |
| 1 | Alice Johnson | 50 | 793.8 |

**Explanation:** The output is derived by applying the required transformations and aggregations to the input data.

### Constraints

- Return only VIP customers (50+ rides AND $500+ spent)
- Sort by `total_spent` in descending order
- Decimal values should be rounded to 2 decimal places
- Handle NULL values gracefully

In [0]:
rides_data = [(1,1,15.50,"2023-01-05",8.5),(2,1,12.30,"2023-01-08",6.2),(3,1,18.75,"2023-01-12",9.3),(4,1,14.20,"2023-01-15",7.1),(5,1,733.05,"2023-01-20",15.0),(6,3,600.00,"2023-02-01",18.0),(7,3,613.25,"2023-02-05",20.0),(8,5,650.40,"2023-03-01",25.0),(9,5,613.50,"2023-03-10",22.0),(10,2,120.50,"2023-01-18",10.0),(11,4,250.00,"2023-02-20",12.0)]
rides_df = spark.createDataFrame(rides_data,["ride_id","user_id","ride_cost","ride_date","distance"])

users_data = [(1,"Alice Johnson","2022-01-15"),(2,"Bob Smith","2021-06-20"),(3,"Carol White","2022-03-10"),(4,"David Brown","2021-12-05"),(5,"Eve Davis","2022-05-18")]
users_df = spark.createDataFrame(users_data,["user_id","name","signup_date"])


grouped_df = (rides_df.
groupBy("user_id")
.agg(
    count("ride_id").alias("total_rides"),
    sum("ride_cost").alias("total_spent")
).
filter((col("total_rides") >= 50) & (col("total_spent") >= 500))
)

output_df = (
grouped_df.alias("g").
join(other=users_df.alias("u"), on=col("g.user_id") == col("u.user_id"), how="inner").
select("u.user_id", "u.name", "g.total_rides", "g.total_spent").
orderBy(col("total_rides").desc())
)

display(output_df)


user_id,name,total_rides,total_spent


## Que 10: Department Salary Aggregation

### Problem

HR needs to analyze compensation across departments to understand salary distribution and headcount.

**Schema columns:** `employees.emp_id`, `employees.name`, `employees.department`, `employees.salary`, `employees.city`

**Output columns:** `department`, `total_salary`, `avg_salary`, `employee_count`

Sort the results by department.

### Examples

#### Example 1

**Input:**

**employees:**

| emp_id | name | department | salary | city |
|--------|------|------------|-------:|------|
| 1 | Employee_1 | Engineering | 76727 | LA |
| 2 | Employee_2 | Finance | 143545 | LA |
| 3 | Employee_3 | HR | 137558 | Boston |
| 4 | Employee_4 | HR | 114215 | LA |

**Output:**

| department | total_salary | avg_salary | employee_count |
|------------|-------------:|-----------:|---------------:|
| Engineering | 289375 | 96458.33 | 3 |
| Finance | 543460 | 108692.0 | 5 |
| HR | 741343 | 123557.17 | 6 |
| Sales | 1074074 | 97643.09 | 11 |

**Explanation:** The output is derived by applying the required transformations and aggregations to the input data.

### Constraints

- Round average salary to exactly 2 decimal places
- Sort by department name alphabetically
- All employees have valid departments and salaries

In [0]:
employees_data = [(1,"Employee_1","Engineering",76727,"LA"),(2,"Employee_2","Finance",143545,"LA"),(3,"Employee_3","HR",137558,"Boston"),(4,"Employee_4","HR",114215,"LA"),(5,"Employee_5","Engineering",100000,"NY"),(6,"Employee_6","Engineering",112648,"Chicago"),(7,"Employee_7","Finance",100000,"NY"),(8,"Employee_8","Finance",100000,"Boston"),(9,"Employee_9","Finance",99915,"LA"),(10,"Employee_10","Finance",100000,"Chicago"),(11,"Employee_11","HR",120000,"NY"),(12,"Employee_12","HR",120000,"Chicago"),(13,"Employee_13","HR",120000,"LA"),(14,"Employee_14","HR",129570,"Boston"),(15,"Employee_15","Sales",90000,"NY"),(16,"Employee_16","Sales",95000,"LA"),(17,"Employee_17","Sales",98000,"Chicago"),(18,"Employee_18","Sales",97000,"Boston"),(19,"Employee_19","Sales",96000,"LA"),(20,"Employee_20","Sales",99000,"NY"),(21,"Employee_21","Sales",100000,"Chicago"),(22,"Employee_22","Sales",101000,"Boston"),(23,"Employee_23","Sales",102000,"LA"),(24,"Employee_24","Sales",98000,"NY"),(25,"Employee_25","Sales",98074,"Chicago")]
employees_df = spark.createDataFrame(employees_data,["emp_id","name","department","salary","city"])

output_df = (
employees_df.
groupBy("department").
agg(
    sum("salary").alias("total_salary"),
    round( avg("salary"), 2 ).alias("avg_salary"),
    count("emp_id").alias("employee_count")
).
orderBy("department")
)

display(output_df)


department,total_salary,avg_salary,employee_count
Engineering,289375,96458.33,3
Finance,543460,108692.0,5
HR,741343,123557.17,6
Sales,1074074,97643.09,11


## Ques 11: User with Most Friends (Bidirectional)

### Problem

A social network stores friendships as a single row per pair, where a friendship between two users means both people are friends with each other. Only friendships whose status is `accepted` count. For each user, their friend count is the number of accepted friendships they take part in, on either side of the pair. Find the users who have the highest friend count, and return each such `user_id` alongside their `friend_count`.

**Schema columns:** `friendships.user_id_1`, `friendships.user_id_2`, `friendships.status`, `friendships.created_at`

**Output columns:** `user_id`, `friend_count`

Sort by the first column in ascending order.

### Examples

#### Example 1

**Input:**

**friendships:**

| user_id_1 | user_id_2 | status | created_at |
|-----------:|-----------:|----------|------------|
| 1 | 2 | accepted | 2024-01-05 |
| 1 | 3 | accepted | 2024-01-10 |
| 2 | 3 | accepted | 2024-01-08 |
| 1 | 4 | pending | 2024-01-15 |
| 2 | 5 | rejected | 2024-01-20 |
| 4 | 5 | accepted | 2024-02-01 |

**Output:**

| user_id | friend_count |
|---------:|-------------:|
| 1 | 2 |
| 2 | 2 |
| 3 | 2 |

**Explanation:**

The accepted friendships are (1,2), (1,3), (2,3) and (4,5). User 1 appears in (1,2) and (1,3), so has 2 friends; users 2 and 3 each appear in two accepted pairs as well, so all three tie at the maximum of 2. Users 4 and 5 only share one accepted friendship each, and the (1,4) pending and (2,5) rejected rows are ignored, so 4 and 5 fall below the maximum and are excluded.

### Constraints

- Only friendships with status `accepted` are counted.
- Each accepted pair adds one friend to both users in the pair.
- Return every user tied for the maximum friend count.
- `friend_count` is an integer.
- Sort by `user_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
friendships_data = [(1,2,"accepted","2024-01-05"), (1,3,"accepted","2024-01-10"), (2,3,"accepted","2024-01-08"), (1,4,"pending","2024-01-15"), (2,5,"rejected","2024-01-20"), (4,5,"accepted","2024-02-01")]

friendships_df = spark.createDataFrame(friendships_data, ["user_id_1","user_id_2","status","created_at"])

filtered_friendship_df = friendships_df.filter(col("status") == "accepted")

merged_df = (
filtered_friendship_df.select(col("user_id_1").alias("user_id")).
union(filtered_friendship_df.select(col("user_id_2").alias("user_id")))
)

grouped_df = (
merged_df.
groupBy("user_id")
.agg(
    count("user_id").alias("friend_count")
))

window_spec = Window.orderBy(col("friend_count").desc())

ranked_df = grouped_df.withColumn("rnk", rank().over(window_spec))

output_df = ranked_df.filter(col("rnk") == 1).drop("rnk")

display(output_df)







user_id,friend_count
1,2
2,2
3,2


## Que 12: Map Filter Reduce Operations

**Difficulty:** Medium

### Problem

You are analyzing raw web traffic to understand how much time visitors spend reading each page. The `web_events` table logs every interaction, but only actual page loads count as reading time. For each page that was loaded at least once, report the number of page loads, the combined reading time, and the average reading time per load. A page load is an event whose type is `page_view`; any other interaction type is ignored.

For each page, output its `page_url`, the count of page-load events as `total_views`, the sum of their `duration_seconds` as `total_duration`, and the average `duration_seconds` per load rounded to 2 decimals as `avg_duration`. List the busiest pages first, ordered by `total_views` from highest to lowest.

**Schema columns:** `web_events.event_id`, `web_events.user_id`, `web_events.page_url`, `web_events.event_type`, `web_events.duration_seconds`, `web_events.event_date`

**Output columns:** `page_url`, `total_views`, `total_duration`, `avg_duration`

Order the result by `total_views DESC`.


### Examples

#### Example 1

**Input:**

**web_events:**

| event_id | user_id | page_url | event_type | duration_seconds | event_date |
|----------:|---------:|----------|------------|-----------------:|------------|
| 1 | 101 | /home | page_view | 45 | 2024-01-01 |
| 3 | 103 | /home | page_view | 50 | 2024-01-01 |
| 8 | 101 | /home | page_view | 55 | 2024-01-03 |
| 2 | 102 | /about | page_view | 32 | 2024-01-01 |
| 7 | 105 | /about | page_view | 28 | 2024-01-03 |
| 4 | 101 | /contact | click | 5 | 2024-01-02 |
| 6 | 104 | /products | page_view | 62 | 2024-01-02 |

**Output:**

| page_url | total_views | total_duration | avg_duration |
|----------|------------:|---------------:|-------------:|
| /home | 3 | 150 | 50.00 |
| /about | 2 | 60 | 30.00 |
| /products | 1 | 62 | 62.00 |

**Explanation:** `/home` has three page_view events (45, 50, 55), so total_views is 3, total_duration is 150, and avg_duration is 150 / 3 = 50.00. The `/contact` click event is not a page_view, so it is dropped and `/contact` never appears in the output.

### Constraints

- Only events with `event_type = 'page_view'` are counted; all other event types are excluded.
- `avg_duration` is the total duration divided by the number of page views, rounded to 2 decimal places.
- A page appears in the output only if it has at least one page_view event.
- Return results matching the expected output schema and order.

In [0]:
web_events_data = [(1,101,"/home","page_view",45,"2024-01-01"), (3,103,"/home","page_view",50,"2024-01-01"), (8,101,"/home","page_view",55,"2024-01-03"), (2,102,"/about","page_view",32,"2024-01-01"), (7,105,"/about","page_view",28,"2024-01-03"), (4,101,"/contact","click",5,"2024-01-02"), (6,104,"/products","page_view",62,"2024-01-02")]

web_events_df = spark.createDataFrame(web_events_data, ["event_id","user_id","page_url","event_type","duration_seconds","event_date"])

filtered_events_df = web_events_df.filter(col("event_type") == "page_view")

grouped_df = (
filtered_events_df.groupBy("page_url").
agg(
    count("page_url").alias("total_views"),
    sum("duration_seconds").alias("total_duration")
))

output_df = (
grouped_df.withColumn("avg_duration", round(col("total_duration")/ col("total_views"), 2))
)

display(output_df)



grouped_df1 = (
filtered_events_df.groupBy("page_url").
agg(
    count(when(col("event_type") == "page_view", "page_url").otherwise('0')).alias("total_views"),
    sum(when(col("event_type") == "page_view", col("duration_seconds")).otherwise(0)).alias("total_duration")
))

display(grouped_df1)

page_url,total_views,total_duration,avg_duration
/home,3,150,50.0
/about,2,60,30.0
/products,1,62,62.0


page_url,total_views,total_duration
/home,3,150
/about,2,60
/products,1,62


## Que 13: Data Quality Validation Report

**Difficulty:** Medium

### Problem

Before trusting a freshly ingested table, an analyst profiles every column to spot missing or inconsistent data. For each column in the `raw_records` table, produce one report row containing:

- Column name
- Total number of rows in the table
- Number of rows where that column is empty
- Percentage of rows where that column is empty, rounded to 1 decimal place
- Number of distinct non-empty values in that column
- The smallest value and the largest value in that column, compared as text

**Schema columns:** `raw_records.id`, `raw_records.name`, `raw_records.email`, `raw_records.age`, `raw_records.salary`, `raw_records.created_date`

**Output columns:** `column_name`, `total_count`, `null_count`, `null_pct`, `distinct_count`, `min_value`, `max_value`


### Examples

#### Example 1

**Input:**

**raw_records:**

| id | name | email | age | salary | created_date |
|---:|----------------|-----------------------|----:|-------:|------------|
| 1 | Alice Johnson | alice@example.com | 28 | 75000 | 2024-01-01 |
| 2 | Bob Smith | | 35 | 85000 | 2024-01-02 |
| 8 | | charlie@example.com | 38 | 92000 | 2024-01-08 |
| 4 | Diana Prince | diana@example.com | | 120000 | 2024-01-04 |
| 6 | Frank Castle | frank@example.com | 45 | | 2024-01-06 |

**Output:**

| column_name | total_count | null_count | null_pct | distinct_count | min_value | max_value |
|-------------|------------:|-----------:|---------:|---------------:|-----------|-----------|
| id | 5 | 0 | 0.0 | 5 | 1 | 8 |
| name | 5 | 1 | 20.0 | 4 | Alice Johnson | Frank Castle |
| email | 5 | 1 | 20.0 | 4 | alice@example.com | frank@example.com |
| age | 5 | 1 | 20.0 | 4 | 28 | 45 |
| salary | 5 | 1 | 20.0 | 4 | 75000 | 120000 |
| created_date | 5 | 0 | 0.0 | 5 | 2024-01-01 | 2024-01-08 |

**Explanation:** The `email` column is empty in 1 of the 5 rows (id 2), so `null_count` is 1 and `null_pct` is 1 / 5 * 100 = 20.0; the 4 remaining values are all distinct, giving `distinct_count` 4, with `alice@example.com` the smallest and `frank@example.com` the largest by text order.

### Constraints

- `null_pct` is rounded to 1 decimal place.
- `distinct_count` counts only non-empty values.
- `min_value` and `max_value` are compared as text.
- One report row is produced per column, in the column order shown in the schema.
- Return results matching the expected output schema and order.

In [0]:
raw_records_data = [(1,"Alice Johnson","alice@example.com",28,75000,"2024-01-01"), (2,"Bob Smith","",35,85000,"2024-01-02"), (8,"","charlie@example.com",38,92000,"2024-01-08"), (4,"Diana Prince","diana@example.com","",120000,"2024-01-04"), (6,"Frank Castle","frank@example.com",45,"","2024-01-06")]

raw_records_df = spark.createDataFrame(raw_records_data, ["id","name","email","age","salary","created_date"])


output = []

for column in raw_records_df.columns:
    current_col = col(column).cast("string")

    result = (
        raw_records_df
        .agg(
            count("*").alias("total_count"),

            count(
                when(
                    current_col.isNull() | (trim(current_col) == ""),
                    1
                )
            ).alias("null_count"),

            round(
                count(
                    when(
                        current_col.isNull() | (trim(current_col) == ""),
                        1
                    )
                ) * 100.0 / count("*"),
                1
            ).alias("null_pct"),

            countDistinct(
                when(
                    current_col.isNotNull() & (trim(current_col) != ""),
                    current_col
                )
            ).alias("distinct_count"),

            min(
                when(
                    current_col.isNotNull() & (trim(current_col) != ""),
                    current_col
                )
            ).alias("min_value"),

            max(
                when(
                    current_col.isNotNull() & (trim(current_col) != ""),
                    current_col
                )
            ).alias("max_value")
        )
        .withColumn("column_name", lit(column))
        .select(
            "column_name",
            "total_count",
            "null_count",
            "null_pct",
            "distinct_count",
            "min_value",
            "max_value"
        )
    )


    output.append(result)
    

fisrt_df = output[0]

for i in range(1, len(output)):
    fisrt_df = fisrt_df.union(output[i])

display(fisrt_df)



column_name,total_count,null_count,null_pct,distinct_count,min_value,max_value
id,5,0,0.0,5,1,8
name,5,1,20.0,4,Alice Johnson,Frank Castle
email,5,1,20.0,4,alice@example.com,frank@example.com
age,5,1,20.0,4,28,45
salary,5,1,20.0,4,120000,92000
created_date,5,0,0.0,5,2024-01-01,2024-01-08


## Que 14: PySpark Custom Aggregation with Statistical Metrics

**Difficulty:** Hard

### Problem

A finance team summarizes transaction amounts by category. For each category, return `txn_count`, `total_amount`, the amount average weighted by `weight` as `weighted_avg_amount`, and the sample standard deviation divided by the ordinary average as `coefficient_of_variation`; round the requested measures as shown and order by category.

**Schema columns:** `transactions.txn_id`, `transactions.category`, `transactions.amount`, `transactions.weight`, `transactions.txn_date`

**Output columns:** `category`, `txn_count`, `total_amount`, `weighted_avg_amount`, `coefficient_of_variation`

Order the result by `category`.

### Examples

#### Example 1

**Input:**

**transactions:**

| txn_id | category | amount | weight | txn_date |
|--------|----------|-------:|-------:|----------|
| T1 | Books | 10 | 1 | 2024-01-01 |
| T2 | Books | 20 | 3 | 2024-01-02 |
| T3 | Games | 30 | 2 | 2024-01-03 |
| T4 | Games | 50 | 2 | 2024-01-04 |

**Output:**

| category | txn_count | total_amount | weighted_avg_amount | coefficient_of_variation |
|----------|----------:|-------------:|--------------------:|-------------------------:|
| Books | 2 | 30.0 | 17.50 | 0.47 |
| Games | 2 | 80.0 | 40.00 | 0.35 |

**Explanation:** For Books, the weighted average is `(10×1 + 20×3) / (1+3) = 17.50`, and the two transactions total `30.0`. Its sample standard deviation divided by the `15` average rounds to `0.47`.

### Constraints

- Weights are positive numeric values.
- Each category has enough observations to compute the requested variability measure.
- Return results matching the expected output schema and order.

In [0]:
transactions_data = [("T1","Books",10,1,"2024-01-01"), ("T2","Books",20,3,"2024-01-02"), ("T3","Games",30,2,"2024-01-03"), ("T4","Games",50,2,"2024-01-04")]

transactions_df = spark.createDataFrame(transactions_data, ["txn_id","category","amount","weight","txn_date"])

display(transactions_df)


grouped_df = (
transactions_df.groupBy("category").agg(
    count(col("txn_id")).alias("txn_count"),
    sum(col("amount")).alias("total_amount"),
    round( sum(col("amount") * col("weight")) / sum(col("weight")) , 2).alias("weighted_avg_amount"),
    round((stddev("amount") / avg("amount")) , 2).alias("coefficient_of_variation")
))

display(grouped_df)

txn_id,category,amount,weight,txn_date
T1,Books,10,1,2024-01-01
T2,Books,20,3,2024-01-02
T3,Games,30,2,2024-01-03
T4,Games,50,2,2024-01-04


category,txn_count,total_amount,weighted_avg_amount,coefficient_of_variation
Books,2,30,17.5,0.47
Games,2,80,40.0,0.35


## Q15: Handling Data Skew in PySpark Aggregations

**Difficulty:** Hard

### Problem

A product analytics team summarizes page-view duration by page. Return `total_views`, distinct viewer count as `unique_users`, two-decimal `avg_duration`, and the continuous median as `median_duration` for each `page_id`, ordered from most to fewest views.

**Schema columns:** `page_views.view_id`, `page_views.page_id`, `page_views.user_id`, `page_views.view_date`, `page_views.duration_ms`

**Output columns:** `page_id`, `total_views`, `unique_users`, `avg_duration`, `median_duration`

Order the result by `total_views DESC`.

### Examples

#### Example 1

**Input:**

**page_views:**

| view_id | page_id | user_id | view_date | duration_ms |
|---------|---------|---------|-----------|------------:|
| V1 | home | U1 | 2024-01-01 | 100 |
| V2 | home | U2 | 2024-01-01 | 300 |
| V3 | home | U1 | 2024-01-02 | 500 |
| V4 | pricing | U3 | 2024-01-02 | 200 |
| V5 | pricing | U4 | 2024-01-03 | 400 |

**Output:**

| page_id | total_views | unique_users | avg_duration | median_duration |
|---------|------------:|-------------:|-------------:|----------------:|
| home | 3 | 2 | 300.00 | 300 |
| pricing | 2 | 2 | 300.00 | 300 |

**Explanation:** The `home` page has durations `100`, `300`, and `500`, so both its average and median are `300`; `U1` appears twice but contributes once to the `2` unique users.

### Constraints

- Every page-view row has a duration in milliseconds.
- Repeated views by the same user count toward views but not toward `unique_users`.
- Pages tied on `total_views` may appear in any order.
- Return results matching the expected output schema and order.

In [0]:
page_views_data = [("V1","home","U1","2024-01-01",100), ("V2","home","U2","2024-01-01",300), ("V3","home","U1","2024-01-02",500), ("V4","pricing","U3","2024-01-02",200), ("V5","pricing","U4","2024-01-03",400)]

page_views_df = spark.createDataFrame(page_views_data, ["view_id","page_id","user_id","view_date","duration_ms"])

grouped_df = (
page_views_df.groupBy("page_id").agg(
    count(col("page_id")).alias("total_views"),
    count_distinct(col("user_id")).alias("unique_users"),
    avg(col("duration_ms")).alias("avg_duration"),
    median(col("duration_ms")).alias("median_duration"),
    sum(col("duration_ms")).alias("total_duration")
))

display(grouped_df)

page_id,total_views,unique_users,avg_duration,median_duration,total_duration
home,3,2,300.0,300.0,900
pricing,2,2,300.0,300.0,600


## Que 16: Kimball Database Design (E-Commerce Star Schema)

**Difficulty:** Hard

### Problem

An analytics team is converting denormalized ecommerce transactions into a fact-style output with integer keys for customers, products, and stores. Assign keys starting at 1 in alphabetical order for each distinct non-null dimension value; every null customer email receives one shared key equal to the highest assigned customer key plus 1. Calculate `revenue` as quantity times unit price rounded to 2 decimals, and order by `txn_id`.

**Schema columns:** `raw_transactions.txn_id`, `raw_transactions.customer_email`, `raw_transactions.product_sku`, `raw_transactions.store_name`, `raw_transactions.txn_date`, `raw_transactions.quantity`, `raw_transactions.unit_price`

**Output columns:** `txn_id`, `customer_key`, `product_key`, `store_key`, `txn_date`, `quantity`, `revenue`

Order the result by `txn_id`.


### Examples

#### Example 1

**Input:**

**raw_transactions:**

| txn_id | customer_email | product_sku | store_name | txn_date | quantity | unit_price |
|-------:|----------------|-------------|------------|----------|---------:|-----------:|
| 1 | NULL | SKU001 | Store_B | 2024-01-02 | 2 | 70.65 |
| 2 | alice@example.com | SKU001 | Store_A | 2024-01-11 | 1 | 71.48 |
| 3 | NULL | SKU001 | Store_A | 2024-01-12 | 1 | 40.82 |
| 4 | adam@example.com | SKU003 | Store_C | 2024-01-14 | 2 | 87.30 |

**Output:**

| txn_id | customer_key | product_key | store_key | txn_date | quantity | revenue |
|-------:|-------------:|------------:|----------:|----------|---------:|--------:|
| 1 | 3 | 1 | 2 | 2024-01-02 | 2 | 141.30 |
| 2 | 2 | 1 | 1 | 2024-01-11 | 1 | 71.48 |
| 3 | 3 | 1 | 1 | 2024-01-12 | 1 | 40.82 |
| 4 | 1 | 2 | 3 | 2024-01-14 | 2 | 174.60 |

**Explanation:** The two shown emails receive keys 1 and 2 alphabetically, while both null emails share key 3.

### Constraints

- Assign each dimension's keys independently from its shown distinct values.
- All null customer emails share the next sequential customer key.
- Return results matching the expected output schema and order.

In [0]:
raw_transactions_data = [(1,None,"SKU001","Store_B","2024-01-02",2,70.65), (2,"alice@example.com","SKU001","Store_A","2024-01-11",1,71.48), (3,None,"SKU001","Store_A","2024-01-12",1,40.82), (4,"adam@example.com","SKU003","Store_C","2024-01-14",2,87.30)]

raw_transactions_df = spark.createDataFrame(raw_transactions_data, ["txn_id","customer_email","product_sku","store_name","txn_date","quantity","unit_price"])

display(raw_transactions_df)


# txn_id	customer_key	product_key	store_key	txn_date	quantity	revenue
customer_window = Window.orderBy(col("customer_email").asc_nulls_last())
product_window = Window.orderBy(col("product_sku").asc_nulls_last())
store_window = Window.orderBy(col("store_name").asc_nulls_last())


ranked_df = (
raw_transactions_df
.withColumn("customer_key", dense_rank().over(customer_window))
.withColumn("product_key", dense_rank().over(product_window))
.withColumn("store_key", dense_rank().over(store_window))
.withColumn("revenue", round((col("quantity") * col("unit_price")), 2))
)

output_df = ranked_df.select("txn_id", "customer_key", "product_key", "store_key", "txn_date", "quantity", "revenue").orderBy("txn_id")
display(output_df)


txn_id,customer_email,product_sku,store_name,txn_date,quantity,unit_price
1,null,SKU001,Store_B,2024-01-02,2,70.65
2,alice@example.com,SKU001,Store_A,2024-01-11,1,71.48
3,null,SKU001,Store_A,2024-01-12,1,40.82
4,adam@example.com,SKU003,Store_C,2024-01-14,2,87.3


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


txn_id,customer_key,product_key,store_key,txn_date,quantity,revenue
1,3,1,2,2024-01-02,2,141.3
2,2,1,1,2024-01-11,1,71.48
3,3,1,1,2024-01-12,1,40.82
4,1,2,3,2024-01-14,2,174.6


## Que 17: Data Warehouse Design for Online Retailer

**Difficulty:** Hard

### Problem

An online retailer is converting raw orders into a fact-style order table with stable numeric identifiers. Assign customer, product, and location IDs in order of first appearance: ascending earliest `order_id` for each customer email, product SKU, and city-state pair. Return one row per raw order with the calendar date, revenue, and an indicator for orders placed on that customer's earliest order date.

**Schema columns:** `raw_orders.order_id`, `raw_orders.customer_email`, `raw_orders.product_sku`, `raw_orders.quantity`, `raw_orders.unit_price`, `raw_orders.order_timestamp`, `raw_orders.shipping_city`, `raw_orders.shipping_state`

**Output columns:** `order_id`, `customer_id`, `product_id`, `location_id`, `order_date`, `quantity`, `revenue`, `is_first_order`

Order the result by `order_id`.


### Examples

#### Example 1

**Input:**

**raw_orders:**

| order_id | customer_email | product_sku | quantity | unit_price | order_timestamp | shipping_city | shipping_state |
|---------:|----------------|-------------|---------:|-----------:|-----------------|---------------|----------------|
| 1001 | alice@example.com | SKU001 | 2 | 49.99 | 2024-01-15 10:30:00 | New York | NY |
| 1002 | bob@example.com | SKU002 | 1 | 99.99 | 2024-01-16 14:20:00 | Los Angeles | CA |
| 1003 | alice@example.com | SKU003 | 3 | 29.99 | 2024-01-17 09:15:00 | New York | NY |

**Output:**

| order_id | customer_id | product_id | location_id | order_date | quantity | revenue | is_first_order |
|---------:|------------:|-----------:|------------:|------------|---------:|--------:|---------------:|
| 1001 | 1 | 1 | 1 | 2024-01-15 | 2 | 99.98 | 1 |
| 1002 | 2 | 2 | 2 | 2024-01-16 | 1 | 99.99 | 1 |
| 1003 | 1 | 3 | 1 | 2024-01-17 | 3 | 89.97 | 0 |

**Explanation:** Alice and New York retain ID `1` on order `1003` because both first appeared on order `1001`.

### Constraints

- `revenue` equals `quantity * unit_price`, rounded to 2 decimal places.
- Set `is_first_order` to `1` when `order_date` equals the customer's earliest `order_date`; multiple orders on that date all receive `1`.
- Sort by `order_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
raw_orders_data = [(1001,"alice@example.com","SKU001",2,49.99,"2024-01-15 10:30:00","New York","NY"), (1002,"bob@example.com","SKU002",1,99.99,"2024-01-16 14:20:00","Los Angeles","CA"), (1003,"alice@example.com","SKU003",3,29.99,"2024-01-17 09:15:00","New York","NY")]

raw_orders_df = spark.createDataFrame(raw_orders_data, ["order_id","customer_email","product_sku","quantity","unit_price","order_timestamp","shipping_city","shipping_state"])

display(raw_orders_df)

# order_id	customer_id	product_id	location_id	order_date	quantity	revenue	is_first_order

customer_window = Window.orderBy(col("customer_email").asc_nulls_last())
product_window = Window.orderBy(col("product_sku").asc_nulls_last())
location_window = Window.orderBy(col("shipping_city").asc_nulls_last())


ranked_df = (
raw_orders_df
.withColumn("customer_id", dense_rank().over(customer_window))
.withColumn("product_id", dense_rank().over(product_window))
.withColumn("location_id", dense_rank().over(location_window))
.withColumn("revenue", round((col("quantity") * col("unit_price")), 2))
.withColumn("order_date", col("order_timestamp").cast("date"))
)

grouped_df = (
raw_orders_df.groupBy("customer_email").agg(
    min(col("order_timestamp")).alias("first_order_date")
))

joined_df = ranked_df.join(grouped_df, on="customer_email")


output_df = (
joined_df.withColumn("is_first_order", when(col("order_timestamp") == col("first_order_date"), lit(1)).otherwise(lit(0)))
.select("order_id", "customer_id", "product_id", "location_id", "order_date", "quantity", "revenue", "is_first_order")
.orderBy("order_id")
)

display(output_df)


order_id,customer_email,product_sku,quantity,unit_price,order_timestamp,shipping_city,shipping_state
1001,alice@example.com,SKU001,2,49.99,2024-01-15 10:30:00,New York,NY
1002,bob@example.com,SKU002,1,99.99,2024-01-16 14:20:00,Los Angeles,CA
1003,alice@example.com,SKU003,3,29.99,2024-01-17 09:15:00,New York,NY


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


order_id,customer_email,product_sku,quantity,unit_price,order_timestamp,shipping_city,shipping_state,customer_id,product_id,location_id,revenue,order_date
1002,bob@example.com,SKU002,1,99.99,2024-01-16 14:20:00,Los Angeles,CA,2,2,1,99.99,2024-01-16
1001,alice@example.com,SKU001,2,49.99,2024-01-15 10:30:00,New York,NY,1,1,2,99.98,2024-01-15
1003,alice@example.com,SKU003,3,29.99,2024-01-17 09:15:00,New York,NY,1,3,2,89.97,2024-01-17


customer_email,first_order_date
alice@example.com,2024-01-15 10:30:00
bob@example.com,2024-01-16 14:20:00


customer_email,order_id,product_sku,quantity,unit_price,order_timestamp,shipping_city,shipping_state,customer_id,product_id,location_id,revenue,order_date,first_order_date
bob@example.com,1002,SKU002,1,99.99,2024-01-16 14:20:00,Los Angeles,CA,2,2,1,99.99,2024-01-16,2024-01-16 14:20:00
alice@example.com,1001,SKU001,2,49.99,2024-01-15 10:30:00,New York,NY,1,1,2,99.98,2024-01-15,2024-01-15 10:30:00
alice@example.com,1003,SKU003,3,29.99,2024-01-17 09:15:00,New York,NY,1,3,2,89.97,2024-01-17,2024-01-15 10:30:00


order_id,customer_id,product_id,location_id,order_date,quantity,revenue,is_first_order
1001,1,1,2,2024-01-15,2,99.98,1
1002,2,2,1,2024-01-16,1,99.99,1
1003,1,3,2,2024-01-17,3,89.97,0


## Que 18: Partition and Clustering Strategy

**Difficulty:** Hard

### Problem

A data platform team wants storage recommendations from query-log patterns. For each `filter_column`, report its query count, average bytes scanned rounded to an integer, and average execution time rounded to 2 decimals. Set `recommended_action` to `partition` when the column name contains `date` and to `cluster` otherwise; order by `query_count` descending, then `filter_column` ascending.

**Schema columns:** `query_log.query_id`, `query_log.table_name`, `query_log.filter_column`, `query_log.filter_value`, `query_log.query_date`, `query_log.scan_bytes`, `query_log.execution_time_ms`

**Output columns:** `filter_column`, `query_count`, `avg_scan_bytes`, `avg_execution_time`, `recommended_action`

Order the result by `query_count DESC`, `filter_column ASC`.

### Examples

#### Example 1

**Input:**

**query_log:**

| query_id | table_name | filter_column | filter_value | query_date | scan_bytes | execution_time_ms |
|---------:|------------|---------------|--------------|------------|-----------:|------------------:|
| 90001 | events | user_id | val_9 | 2024-02-21 | 34147728 | 3195 |
| 90002 | orders | date | val_19 | 2024-03-06 | 26737303 | 4575 |
| 90003 | orders | user_id | val_10 | 2024-03-06 | 1990773 | 2708 |
| 90004 | users | event_type | val_3 | 2024-03-05 | 48002178 | 4946 |
| 90005 | users | event_type | val_4 | 2024-02-22 | 39839397 | 1715 |

**Output:**

| filter_column | query_count | avg_scan_bytes | avg_execution_time | recommended_action |
|---------------|------------:|---------------:|-------------------:|--------------------|
| event_type | 2 | 43920788 | 3330.50 | cluster |
| user_id | 2 | 18069251 | 2951.50 | cluster |
| date | 1 | 26737303 | 4575.00 | partition |

**Explanation:** The two repeated columns appear first, and only the date-like column receives `partition`.

### Constraints

- Match `date` case-insensitively within the column name.
- Include columns regardless of query frequency.
- Return results matching the expected output schema and order.

In [0]:
query_log_data = [(90001,"events","user_id","val_9","2024-02-21",34147728,3195), (90002,"orders","date","val_19","2024-03-06",26737303,4575), (90003,"orders","user_id","val_10","2024-03-06",1990773,2708), (90004,"users","event_type","val_3","2024-03-05",48002178,4946), (90005,"users","event_type","val_4","2024-02-22",39839397,1715)]

query_log_df = spark.createDataFrame(query_log_data, ["query_id","table_name","filter_column","filter_value","query_date","scan_bytes","execution_time_ms"])

display(query_log_df)

# filter_column	query_count	avg_scan_bytes	avg_execution_time	recommended_action
grouped_df = (
query_log_df.groupBy("filter_column").agg(
    count("*").alias("query_count"),
    round(avg(col("scan_bytes")), 2).alias("avg_scan_bytes"),
    round(avg(col("execution_time_ms")), 2).alias("avg_execution_time")
))

output_df = (
grouped_df.withColumn("recommended_action", when(col("filter_column").contains("date"), "partition").otherwise("cluster"))
.orderBy(col("query_count").desc(), col("filter_column"))
)

display(output_df)




query_id,table_name,filter_column,filter_value,query_date,scan_bytes,execution_time_ms
90001,events,user_id,val_9,2024-02-21,34147728,3195
90002,orders,date,val_19,2024-03-06,26737303,4575
90003,orders,user_id,val_10,2024-03-06,1990773,2708
90004,users,event_type,val_3,2024-03-05,48002178,4946
90005,users,event_type,val_4,2024-02-22,39839397,1715


filter_column,query_count,avg_scan_bytes,avg_execution_time,recommended_action
event_type,2,4.39207875E7,3330.5,cluster
user_id,2,1.80692505E7,2951.5,cluster
date,1,2.6737303E7,4575.0,partition


## Que 19: Index Strategy for Performance

**Difficulty:** Hard

### Problem

A database team wants to prioritize columns referenced in slow-query filters, relationship matches, and requested result ordering. For each table, column, and usage type, return the number of query appearances and average execution time. Define `priority_score` as `frequency * avg_execution_time / 1000`, rounded to 2 decimals; order by score descending, then usage type in the order `order`, `where`, `join`, then `column_name` ascending.

**Schema columns:** `slow_queries.query_id`, `slow_queries.table_name`, `slow_queries.where_columns`, `slow_queries.join_columns`, `slow_queries.order_columns`, `slow_queries.execution_time_ms`, `slow_queries.row_count`

**Output columns:** `table_name`, `column_name`, `usage_type`, `frequency`, `avg_execution_time`, `priority_score`

Order the result by `priority_score DESC`, `usage_type (order → where → join)`, `column_name ASC`.


### Examples

#### Example 1

**Input:**

**slow_queries:**

| query_id | table_name | where_columns | join_columns | order_columns | execution_time_ms | row_count |
|---------:|------------|---------------|--------------|---------------|------------------:|----------:|
| 1 | orders | status,amount | customer_id,product_id | order_date,amount | 12000 | 250000 |
| 2 | orders | status | customer_id | NULL | 8000 | 180000 |
| 3 | orders | amount,cust_name | NULL | order_date | 10000 | 220000 |

**Output:**

| table_name | column_name | usage_type | frequency | avg_execution_time | priority_score |
|------------|-------------|------------|----------:|-------------------:|---------------:|
| orders | order_date | order | 2 | 11000 | 22.00 |
| orders | amount | where | 2 | 11000 | 22.00 |
| orders | customer_id | join | 2 | 10000 | 20.00 |
| orders | status | where | 2 | 10000 | 20.00 |
| orders | product_id | join | 1 | 12000 | 12.00 |
| orders | cust_name | where | 1 | 10000 | 10.00 |


**Explanation:** Equal scores use the documented usage-type order and then the column name.

### Constraints

- Split comma-separated columns and ignore null or empty lists.
- Round `priority_score` to 2 decimal places.
- Return results matching the expected output schema and order.

In [0]:
slow_queries_data = data = [(1, "orders", "status,amount", "customer_id,product_id", "order_date,amount", 12000, 250000),(2, "orders", "status", "customer_id", None, 8000, 180000),(3, "orders", "amount,cust_name", None, "order_date", 10000, 220000),]

slow_queries_df = spark.createDataFrame(slow_queries_data, ["query_id","table_name","where_columns","join_columns","order_columns","execution_time_ms","row_count"])


display(slow_queries_df)

where_column_df = (
slow_queries_df
.withColumn("column_name", explode(split(col("where_columns"), ",")))
.withColumn("usage_type", lit("where"))
.select("query_id", "table_name", "column_name", "usage_type", "execution_time_ms")
)

join_column_df = (
slow_queries_df
.withColumn("column_name", explode(split(col("join_columns"), ",")))
.withColumn("usage_type", lit("join"))
.select("query_id", "table_name", "column_name", "usage_type", "execution_time_ms")
)

order_column_df = (
slow_queries_df
.withColumn("column_name", explode(split(col("order_columns"), ",")))
.withColumn("usage_type", lit("order"))
.select("query_id", "table_name", "column_name", "usage_type", "execution_time_ms")
)

combined_df = where_column_df.unionAll(join_column_df).unionAll(order_column_df)


grouped_df = (
combined_df.groupBy("table_name", "column_name", "usage_type").agg(
    count("*").alias("frequency"),
    round(avg(col("execution_time_ms")), 2).alias("avg_execution_time")
))


priority_score_df = grouped_df.withColumn("priority_score", round((col("frequency") * col("avg_execution_time")) / 1000, 2))


custom_order_by_column_df = (
priority_score_df.withColumn("custom_order_by_column", (
    when(col("usage_type") == "order", 1).
    when(col("usage_type") == "where", 2).
    otherwise(3)
)))


final_df = (
    custom_order_by_column_df.
    orderBy(col("priority_score").desc(), col("custom_order_by_column"), col("column_name"))
    .drop(col("custom_order_by_column"))
)



display(final_df)

query_id,table_name,where_columns,join_columns,order_columns,execution_time_ms,row_count
1,orders,"status,amount","customer_id,product_id","order_date,amount",12000,250000
2,orders,status,customer_id,null,8000,180000
3,orders,"amount,cust_name",null,order_date,10000,220000


table_name,column_name,usage_type,frequency,avg_execution_time,priority_score
orders,order_date,order,2,11000.0,22.0
orders,amount,where,2,11000.0,22.0
orders,status,where,2,10000.0,20.0
orders,customer_id,join,2,10000.0,20.0
orders,amount,order,1,12000.0,12.0
orders,product_id,join,1,12000.0,12.0
orders,cust_name,where,1,10000.0,10.0


## Top Customers by Lifetime Value

**Difficulty:** Hard

### Problem

An analytics team wants a reusable customer-value summary from transaction history. For each customer, calculate lifetime spend, transaction count, and the category with the greatest spend, then return the five customers with the greatest lifetime value.

**Schema columns:** `transactions.txn_id`, `transactions.customer_id`, `transactions.product_category`, `transactions.amount`, `transactions.txn_date`

**Output columns:** `customer_id`, `lifetime_value`, `top_category`, `top_category_amount`, `txn_count`

Order the result by `lifetime_value DESC`, `customer_id ASC`, and return at most 5 rows.

### Examples

#### Example 1

**Input:**

**transactions:**

| txn_id | customer_id | product_category | amount | txn_date |
|-------:|-------------|------------------|-------:|----------|
| 1 | C001 | Electronics | 250.00 | 2024-01-05 |
| 2 | C001 | Clothing | 75.50 | 2024-01-10 |
| 3 | C001 | Electronics | 120.00 | 2024-01-15 |
| 16 | C001 | Home | 180.00 | 2024-01-25 |

**Output:**

| customer_id | lifetime_value | top_category | top_category_amount | txn_count |
|-------------|---------------:|--------------|--------------------:|----------:|
| C001 | 625.50 | Electronics | 370.00 | 4 |

**Explanation:** C001 spends 625.50 overall, including a leading 370.00 in Electronics, across four transactions.

### Constraints

- `top_category` is the category with the greatest summed amount for that customer; if categories tie, choose the alphabetically smallest `product_category`.
- Sort by `lifetime_value` descending, then `customer_id` ascending, and return at most five rows.
- Return results matching the expected output schema and order.

In [0]:
transactions_data = [(1,"C001","Electronics",250.00,"2024-01-05"), (2,"C001","Clothing",75.50,"2024-01-10"), (3,"C001","Electronics",120.00,"2024-01-15"), (16,"C001","Home",180.00,"2024-01-25")]

transactions_df = spark.createDataFrame(transactions_data, ["txn_id","customer_id","product_category","amount","txn_date"])

display(transactions_df)

# customer_id	lifetime_value	top_category	top_category_amount	txn_count
grouped_df_customer = (
transactions_df.groupBy("customer_id").agg(
    sum(col("amount")).alias("lifetime_value"),
    count(col("txn_id")).alias("txn_count")
))

grouped_df_customer_product = (
transactions_df
.groupBy("customer_id", "product_category").agg(
    sum(col("amount")).alias("product_total_amount"),
))


window_spec = Window.partitionBy("customer_id").orderBy(col("product_total_amount").desc(), col("product_category"))

ranked_df = grouped_df_customer_product.withColumn("rank", row_number().over(window_spec)).filter(col("rank") == 1)

output_df = (
grouped_df_customer.alias("g").join(ranked_df.alias("r"), on=col("r.customer_id") == col("g.customer_id"), how="inner")
.select("g.*", col("r.product_category").alias("top_category"), col("r.product_total_amount").alias("top_category_amount"))
.orderBy(col("g.lifetime_value").desc(), col("g.customer_id"))
.limit(5)
)


display(output_df)



txn_id,customer_id,product_category,amount,txn_date
1,C001,Electronics,250.0,2024-01-05
2,C001,Clothing,75.5,2024-01-10
3,C001,Electronics,120.0,2024-01-15
16,C001,Home,180.0,2024-01-25


customer_id,lifetime_value,txn_count,top_category,top_category_amount
C001,625.5,4,Electronics,370.0
